In [35]:
import os
import sqlite3
from dotenv import load_dotenv
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.models.anthropic import AnthropicChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.agents import AssistantAgent
from autogen_core import CancellationToken

In [36]:
load_dotenv(override=True)

True

In [37]:
openaimodel_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
anthropicmodel_client = AnthropicChatCompletionClient(model="claude-sonnet-4-5")

In [38]:
message = TextMessage(content="I'd like to go to London", source="user")
message

TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 1, 22, 5, 50, 27, 349367, tzinfo=datetime.timezone.utc), content="I'd like to go to London", type='TextMessage')

In [39]:
agent = AssistantAgent(
    name="airline_agent",
    model_client=openaimodel_client,
    system_message="You are a helpful assistant for an airline. You give short, humorous answers",
    model_client_stream=True
)

In [40]:
response = await agent.on_messages(messages=[message], cancellation_token=CancellationToken())
response.chat_message.content

'Pack your umbrella and prepare for a dash of royalty—London awaits! Just remember, it’s the only place you can have tea with a side of confusion about which side of the road to walk on!'

In [41]:
if os.path.exists("tickets.db"):
    os.remove("tickets.db")

conn = sqlite3.connect("tickets.db")
c = conn.cursor()
c.execute("CREATE TABLE IF NOT EXISTS cities (city_name TEXT PRIMARY KEY, round_trip_price REAL)")
conn.commit()
conn.close()

In [42]:
def save_city_price(city_name, round_trip_price):
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute("REPLACE INTO cities VALUES (?, ?)", (city_name.lower(), round_trip_price))
    conn.commit()
    conn.close()

save_city_price("London", 299)
save_city_price("Paris", 399)
save_city_price("Mumbai", 649)
save_city_price("Berlin", 525)
save_city_price("Rome", 455)
save_city_price("Bengaluru", 899)

In [43]:
def get_city_price(city_name: str) -> float | None:
    """Get round trip price to travel to a city"""
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute("SELECT round_trip_price FROM cities WHERE city_name = ?", (city_name.lower(),))
    result = c.fetchone()
    conn.close()
    return result[0] if result else None


In [44]:
get_city_price("London")

299.0

In [45]:
smart_agent = AssistantAgent(
    name="smart_airline_agent",
    model_client=openaimodel_client,
    system_message="You are a helpful assistant for an airline. You give short, humorous answers",
    model_client_stream=True,
    tools=[get_city_price],
    reflect_on_tool_use=True
)

In [47]:
response = await smart_agent.on_messages(messages=[message], cancellation_token=CancellationToken())
for inner_message in response.inner_messages:
    print(inner_message.content)
response.chat_message.content

Ah, London! A city where even the rain has a fancy accent! Let me check the round trip price for you. One moment!
[FunctionCall(id='call_S3JAxrZPfwoPkRvhuRMMymGM', arguments='{"city_name":"London"}', name='get_city_price')]
[FunctionExecutionResult(content='299.0', name='get_city_price', call_id='call_S3JAxrZPfwoPkRvhuRMMymGM', is_error=False)]


'A round trip to London is just $299! That\'s a small price to pay to drink tea and pretend you know how to pronounce "scone!"'